In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import SimpleITK as sitk
import time
import pydicom


In [2]:
series_uids = []
series_list_file = '/data/zhangwd/data/examples/brain/bone_removed/removed_dicom.txt'
with open(series_list_file) as f:
    for line in f.readlines():
        line = line.strip()
        if line is None or len(line) == 0:
            continue
        series_uids.append(line)
series_uid = series_uids[0]
print(series_uid)

/data/zhangwd/data/examples/brain/bystudy/1.3.12.2.1107.5.1.4.60320.30000016072900171834200000026/1.3.12.2.1107.5.99.2.9594.30000016072913081812500000799


In [3]:
reader = sitk.ImageSeriesReader()
filenamesDICOM = reader.GetGDCMSeriesFileNames(series_uid)
reader.SetFileNames(filenamesDICOM)
imgOriginal = reader.Execute()

series_uid_window = '/data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20170320003131115/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240'
single_dcm = pydicom.read_file(filenamesDICOM[0])

In [4]:
# tmp = sitk.GetArrayFromImage(imgOriginal)
# tmp.shape
# new_img = sitk.GetImageFromArray(tmp)
# new_img.SetSpacing([2.5,3.5,4.5])

# directory = './test'
# os.makedirs(directory, exist_ok=True)

# writer = sitk.ImageFileWriter()
# # Use the study/series/frame of reference information given in the meta-data
# # dictionary and not the automatically generated information from the file IO
# writer.KeepOriginalImageUIDOn()

# # Copy relevant tags from the original meta-data dictionary (private tags are also
# # accessible).
# tags_to_copy = ["0010|0010", # Patient Name
#                 "0010|0020", # Patient ID
#                 "0010|0030", # Patient Birth Date
#                 "0020|000D", # Study Instance UID, for machine consumption
#                 "0020|0010", # Study ID, for human consumption
#                 "0008|0020", # Study Date
#                 "0008|0030", # Study Time
#                 "0008|0050", # Accession Number
#                 "0008|0060"  # Modality
# ]

# modification_time = time.strftime("%H%M%S")
# modification_date = time.strftime("%Y%m%d")


# # Copy some of the tags and add the relevant tags indicating the change.
# # For the series instance UID (0020|000e), each of the components is a number, cannot start
# # with zero, and separated by a '.' We create a unique series ID using the date and time.
# # tags of interest:
# direction = new_img.GetDirection()
# print(new_img.HasMetaDataKey("0008|0021"))
# series_tag_values = [(k, new_img.GetMetaData(k)) for k in tags_to_copy if new_img.HasMetaDataKey(k)] + \
#                   [("0008|0031",modification_time), # Series Time
#                   ("0008|0021",modification_date), # Series Date
#                   ("0008|0008","DERIVED\\SECONDARY"), # Image Type
#                   ("0020|000e", "1.2.826.0.1.3680043.2.1125."+modification_date+".1"+modification_time), # Series Instance UID
#                   ("0020|0037", '\\'.join(map(str, (direction[0], direction[3], direction[6],# Image Orientation (Patient)
#                                                     direction[1],direction[4],direction[7])))),
#                   ("0008|103e", "Created-SimpleITK")] # Series Description
# print(new_img.GetMetaDataKeys())
# for i in range(new_img.GetDepth()):
#     image_slice = new_img[:,:,i]
#     # Tags shared by the series.
#     for tag, value in series_tag_values:
#         image_slice.SetMetaData(tag, value)
#     # Slice specific tags.
#     image_slice.SetMetaData("0008|0012", time.strftime("%Y%m%d")) # Instance Creation Date
#     image_slice.SetMetaData("0008|0013", time.strftime("%H%M%S")) # Instance Creation Time
#     image_slice.SetMetaData("0008|0060", "CT")  # set the type to CT so the thickness is carried over
#     image_slice.SetMetaData("0020|0032", '\\'.join(map(str,new_img.TransformIndexToPhysicalPoint((0,0,i))))) # Image Position (Patient)
#     image_slice.SetMetaData("0020,0013", str(i)) # Instance Number

#     # Write to the output directory and add the extension dcm, to force writing in DICOM format.
#     writer.SetFileName(os.path.join(directory,str(i)+'.dcm'))
#     writer.Execute(image_slice)
# print(new_img.GetMetaDataKeys())

In [5]:
def save_threshold_dicom(threshold_img, outdir):
    tmp = sitk.GetArrayFromImage(threshold_img)
    new_img = sitk.GetImageFromArray(tmp)
    new_img.SetSpacing([2.5,3.5,4.5])

    directory = outdir
    os.makedirs(directory, exist_ok=True)

    writer = sitk.ImageFileWriter()
    # Use the study/series/frame of reference information given in the meta-data
    # dictionary and not the automatically generated information from the file IO
    writer.KeepOriginalImageUIDOn()

    # Copy relevant tags from the original meta-data dictionary (private tags are also
    # accessible).
    tags_to_copy = ["0010|0010", # Patient Name
                    "0010|0020", # Patient ID
                    "0010|0030", # Patient Birth Date
                    "0020|000D", # Study Instance UID, for machine consumption
                    "0020|0010", # Study ID, for human consumption
                    "0008|0020", # Study Date
                    "0008|0030", # Study Time
                    "0008|0050", # Accession Number
                    "0008|0060"  # Modality
    ]

    modification_time = time.strftime("%H%M%S")
    modification_date = time.strftime("%Y%m%d")


    # Copy some of the tags and add the relevant tags indicating the change.
    # For the series instance UID (0020|000e), each of the components is a number, cannot start
    # with zero, and separated by a '.' We create a unique series ID using the date and time.
    # tags of interest:
    direction = new_img.GetDirection()
    print(new_img.HasMetaDataKey("0008|0021"))
    series_tag_values = [(k, new_img.GetMetaData(k)) for k in tags_to_copy if new_img.HasMetaDataKey(k)] + \
                      [("0008|0031",modification_time), # Series Time
                      ("0008|0021",modification_date), # Series Date
                      ("0008|0008","DERIVED\\SECONDARY"), # Image Type
                      ("0020|000e", "1.2.826.0.1.3680043.2.1125."+modification_date+".1"+modification_time), # Series Instance UID
                      ("0020|0037", '\\'.join(map(str, (direction[0], direction[3], direction[6],# Image Orientation (Patient)
                                                        direction[1],direction[4],direction[7])))),
                      ("0008|103e", "Created-SimpleITK")] # Series Description
    print(new_img.GetMetaDataKeys())
    for i in range(new_img.GetDepth()):
        image_slice = new_img[:,:,i]
        # Tags shared by the series.
        for tag, value in series_tag_values:
            image_slice.SetMetaData(tag, value)
        # Slice specific tags.
        image_slice.SetMetaData("0008|0012", time.strftime("%Y%m%d")) # Instance Creation Date
        image_slice.SetMetaData("0008|0013", time.strftime("%H%M%S")) # Instance Creation Time
        image_slice.SetMetaData("0008|0060", "CT")  # set the type to CT so the thickness is carried over
        image_slice.SetMetaData("0020|0032", '\\'.join(map(str,new_img.TransformIndexToPhysicalPoint((0,0,i))))) # Image Position (Patient)
        image_slice.SetMetaData("0020,0013", str(i)) # Instance Number

        # Write to the output directory and add the extension dcm, to force writing in DICOM format.
        writer.SetFileName(os.path.join(directory,str(i)+'.dcm'))
        writer.Execute(image_slice)
    print(new_img.GetMetaDataKeys())

In [6]:
wl = 287
ww = 332
lower = int(wl-ww/2)
upper = int(wl+ww/2)

In [7]:
def save_raw_threshold_result(lower, upper, imgOriginal, outdir):
#     image2DThresh = sitk.Threshold(imgOriginal, lower=lower, upper=upper)
    
    threshold_filter = sitk.BinaryThresholdImageFilter()
    threshold_filter.SetInsideValue(1)
    threshold_filter.SetOutsideValue(0)
    threshold_filter.SetLowerThreshold(lower)
    threshold_filter.SetUpperThreshold(2000)
    threshold_img = threshold_filter.Execute(imgOriginal)
    
    erode_filter = sitk.BinaryErodeImageFilter()
    erode_filter.SetKernelRadius(1)
    vessel_erode_img = erode_filter.Execute(threshold_img)
    
    dilation_filter = sitk.BinaryDilateImageFilter()
    dilation_filter.SetKernelRadius(1)
    vessel_dilation_img = dilation_filter.Execute(vessel_erode_img)
#     cca = sitk.ConnectedComponentImageFilter()
#     cca_image = cca.Execute(image2DThresh)
#     os.makedirs('./threshold', exist_ok=True)
#     save_threshold_dicom(image2DThresh, outdir)
    os.makedirs('{}/raw'.format(outdir), exist_ok=True)
    os.makedirs('{}/dilation'.format(outdir), exist_ok=True)
    sitk.WriteImage(threshold_img, '{}/raw/threshold_{}_{}.nii.gz'.format(outdir, lower, upper))
    sitk.WriteImage(vessel_dilation_img, '{}/dilation/threshold_{}_{}.nii.gz'.format(outdir, lower, upper))
    
def save_threshold_result(series_uid, ww, wl, outdir):
    print('====> processing {}'.format(series_uid))
    reader = sitk.ImageSeriesReader()
    filenamesDICOM = reader.GetGDCMSeriesFileNames(series_uid)
    reader.SetFileNames(filenamesDICOM)
    imgOriginal = reader.Execute()
    
    lower = int(wl-ww/2)
    upper = int(wl+ww/2)
    
    os.makedirs(outdir, exist_ok=True)
    outdir = os.path.join(outdir, os.path.basename(series_uid))
    os.makedirs(outdir, exist_ok=True)
    save_raw_threshold_result(lower, upper, imgOriginal, outdir)
    

In [8]:
outdir = './threshold'
for series_uid in series_uids:
    beg = time.time()
    save_threshold_result(series_uid, ww, wl, outdir)
    end = time.time()
    print('time elapsed:\t{:.3f}'.format(end-beg))

====> processing /data/zhangwd/data/examples/brain/bystudy/1.3.12.2.1107.5.1.4.60320.30000016072900171834200000026/1.3.12.2.1107.5.99.2.9594.30000016072913081812500000799
time elapsed:	31.833
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20170320003131115/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240
time elapsed:	38.255
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20181017005827621/1.3.12.2.1107.5.99.2.9594.30000018101817420531200000128
time elapsed:	38.754
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20160408005129972/1.3.12.2.1107.5.99.2.9594.30000016040823144126500004946
time elapsed:	35.461
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20190505015817095/1.3.12.2.1107.5.99.2.9594.30000019042916441620300004106
time elapsed:	49.127
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20190401064814798/1.3.1